# FinAdvisor paper runs on Kaggle (two T4 GPUs, two runs at a time)

Kaggle runs its share of the paper runs (step 1) while Colab runs the rest (`colab_runner.ipynb`). A run is never
split between the two, so each run's rows come from one machine type and one commit.

**First session**
1. Settings (right panel): **Accelerator → GPU T4 x2** (not P100: the other runs used a T4) and **Internet → On**
   (Internet needs a phone-verified Kaggle account).
2. **Save Version → Save & Run All (Commit)**. It runs in the background; the browser can be closed. The notebook
   stops itself after about 11 h 15 min so Kaggle saves the results before its 12-hour limit.

**Every later session**
1. **Add Input → Your Work → Notebooks →** this notebook, so the last session's results come back. In the Input panel,
   make sure it points at the **latest version**.
2. **Save Version → Save & Run All (Commit)** again. Step 3 prints what it restored; every run resumes where it stopped.

The results are in the version's **Output** tab (`finadvisor-results/`). Download them from there.

## 1. Settings

In [ ]:
BRANCH = "paper-experiments"
REPO = "https://github.com/b-rahul07/FinAdv.git"
# Kaggle's share, in this order. Colab runs every other paper run (colab_runner.ipynb, USE_KAGGLE = True there).
# Never run one of these on Colab as well.
KAGGLE_RUNS = ["B1_plain_llm", "B3_rules_in_prompt", "A1T_full_turn", "ADV_adversarial", "B2_llm_rag",
               "A6_no_pressure_fence", "A4_no_number_guard", "M1_qwen2.5-7b", "M1_phi4-mini", "M1_llama3.2-3b"]
MODELS = ["qwen2.5:3b", "qwen2.5:7b", "phi4-mini", "llama3.2:3b"]
STOP_AFTER_HOURS = 11.25   # Kaggle ends a session at 12 h; stopping first lets the version save its output

## 2. GPUs

In [ ]:
import subprocess, time
SESSION_START = time.time()
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4 x2 (and Internet -> On), then run again.")
GPUS = [line.strip() for line in gpu.stdout.strip().splitlines()]
print("GPUs:", GPUS)
if not all("T4" in g for g in GPUS):
    raise SystemExit("Choose GPU T4 x2: the paper's other runs used a T4, and latency is reported.")

## 3. Results (restored from the last session's output)

In [ ]:
import glob, json, os, shutil
WORK = "/kaggle/working/finadvisor-results"
os.makedirs(f"{WORK}/paper", exist_ok=True)
os.makedirs(f"{WORK}/logs", exist_ok=True)
os.makedirs(f"{WORK}/platform", exist_ok=True)

def repair(path):
    """Drop a torn last line (a session stopped mid-write), so the next row starts on a line of its own."""
    if not os.path.exists(path):
        return
    with open(path, "rb+") as f:
        data = f.read()
        if data and not data.endswith(b"\n"):
            f.truncate(data.rfind(b"\n") + 1)

def valid_rows(path):
    if not os.path.exists(path):
        return []
    rows = []
    for line in open(path, encoding="utf-8"):
        try:
            rows.append(json.loads(line))
        except ValueError:
            pass
    return rows

# The largest copy of each file wins (an older input version never replaces newer rows).
for sub in ("paper", "logs", "platform"):
    for src in glob.glob(f"/kaggle/input/**/finadvisor-results/{sub}/*", recursive=True):
        dst = f"{WORK}/{sub}/{os.path.basename(src)}"
        if os.path.isfile(src) and (not os.path.exists(dst) or os.path.getsize(src) > os.path.getsize(dst)):
            shutil.copy2(src, dst)
for path in sorted(glob.glob(f"{WORK}/paper/*.jsonl")):
    repair(path)
    print(f"restored {os.path.basename(path):28s} {len(valid_rows(path)):4d} rows")
if not glob.glob(f"{WORK}/paper/*.jsonl"):
    print("no earlier results (first session, or the last session's output was not added as input)")

## 4. Code

In [ ]:
ROOT = "/tmp/FinAdv"

def sh(cmd, check=True, quiet=False):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if not quiet and r.stdout.strip():
        print(r.stdout.strip())
    if check and r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:])
        raise SystemExit(f"failed: {cmd}")
    return r

if not os.path.isdir(f"{ROOT}/.git"):
    sh(f"git clone -q -b {BRANCH} {REPO} {ROOT}")
sh(f"git -C {ROOT} fetch -q origin {BRANCH}")
sh(f"git -C {ROOT} reset -q --hard origin/{BRANCH}")
LATEST = sh(f"git -C {ROOT} rev-parse HEAD", quiet=True).stdout.strip()
ALIASES = json.load(open(f"{ROOT}/experiments/commit_aliases.json"))["aliases"]
FIXES = "a53be87"   # 11 Nov 2026: wish in the audit, number words, router. A6, A1T, A4 and ADV need it.
print("code:", sh(f"git -C {ROOT} log --oneline -1", quiet=True).stdout.strip())

## 5. Python packages

In [ ]:
import importlib.util
if importlib.util.find_spec("finadvisor") is None or importlib.util.find_spec("chromadb") is None:
    print("installing (a few minutes; warnings about Kaggle's own packages are harmless) ...")
    sh(f"pip install -q -e '{ROOT}[rag]'", quiet=True)
    importlib.invalidate_caches()
print("packages ok")

## 6. Ollama: one server per GPU

Server *i* sees only GPU *i* and listens on port 11434 + *i*. Each holds one model at a time.

In [ ]:
import signal, urllib.request
SERVERS = {}

def port(i):
    return 11434 + i

def server_up(i):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{port(i)}/api/tags", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

def log_tail(i, n=40):
    try:
        return "".join(open(f"/tmp/ollama-{i}.log").readlines()[-n:])
    except FileNotFoundError:
        return "(no Ollama log)"

def start_server(i):
    if server_up(i):
        return
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(i), "OLLAMA_HOST": f"127.0.0.1:{port(i)}",
           "OLLAMA_MAX_LOADED_MODELS": "1", "OLLAMA_NUM_PARALLEL": "1"}
    SERVERS[i] = subprocess.Popen(["ollama", "serve"], stdout=open(f"/tmp/ollama-{i}.log", "a"),
                                  stderr=subprocess.STDOUT, env=env, start_new_session=True)
    for _ in range(60):
        if server_up(i):
            print(f"Ollama {i} is running on GPU {i}")
            return
        time.sleep(1)
    print(log_tail(i))
    raise RuntimeError(f"Ollama {i} did not start")

def restart_server(i):
    p = SERVERS.get(i)
    if p and p.poll() is None:
        os.killpg(p.pid, signal.SIGTERM)
        p.wait(timeout=30)
    time.sleep(5)
    start_server(i)

if not shutil.which("ollama"):
    print("installing Ollama ...")
    sh("apt-get install -y -qq zstd", quiet=True, check=False)
    sh("curl -fsSL https://ollama.com/install.sh | sh", quiet=True)
for i in range(len(GPUS)):
    start_server(i)

## 7. Models (about 10 GB, once per session; both servers share them)

In [ ]:
def pulled(i=0):
    with urllib.request.urlopen(f"http://127.0.0.1:{port(i)}/api/tags", timeout=10) as r:
        return {m["name"] for m in json.load(r)["models"]}

for m in MODELS:
    if m in pulled() or f"{m}:latest" in pulled():
        print("have", m)
        continue
    print("pulling", m, "...", flush=True)
    r = subprocess.run(["ollama", "pull", m], capture_output=True, text=True,
                       env={**os.environ, "OLLAMA_HOST": f"127.0.0.1:{port(0)}"})
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise SystemExit(f"could not pull {m}")
for i in range(len(GPUS)):
    print(f"server {i} models:", sorted(pulled(i)))

## 8. Runs (two at a time, one per GPU)

Each run gets its own copy of the code at its commit: a new run starts on the latest commit; a run started in an
earlier session finishes on the commit it started on. A6, A1T, A4 and ADV are never run on code older than the
11 Nov 2026 fixes. At the time limit the runs are stopped between personas' rows and resume next session.

In [ ]:
import sys, threading, yaml

NEED_FIXES = ["A6_no_pressure_fence", "A1T_full_turn", "A4_no_number_guard", "ADV_adversarial"]
DEADLINE = SESSION_START + STOP_AFTER_HOURS * 3600
STOP = threading.Event()
LOCK = threading.Lock()
CLAIMED, PROCS, PROBLEMS = set(), {}, []

def run_status(run_id):
    """(commit of the first row, rows, rows wanted, file) for one run."""
    cfg = yaml.safe_load(open(f"{ROOT}/experiments/configs/paper/{run_id}.yaml"))
    want = 50 * 3 * cfg.get("per_level", 1) if run_id.startswith("ADV") else 240 * cfg.get("replicates", 1)
    path = f"{WORK}/paper/{os.path.basename(cfg['output'])}"
    repair(path)
    rows = valid_rows(path)
    first = rows[0].get("git_commit") if rows else None
    return ALIASES.get(first, first), len(rows), want, path

def has_fixes(commit):
    return subprocess.run(["git", "-C", ROOT, "merge-base", "--is-ancestor", FIXES, commit]).returncode == 0

def code_for(run_id, commit):
    dest = f"/tmp/code/{run_id}"
    if not os.path.isdir(f"{dest}/.git"):
        sh(f"git clone -q --shared {ROOT} {dest}")
    sh(f"git -C {dest} checkout -q --detach {commit}")
    if not os.path.islink(f"{dest}/results"):
        os.symlink(WORK, f"{dest}/results")
    if run_id.startswith("B2") and not os.path.isdir(f"{dest}/data/chroma_db"):
        sh(f"cd {dest} && PYTHONPATH={dest} {sys.executable} scripts/build_index.py", quiet=True)
    return dest

def module_for(run_id):
    if run_id.startswith("B"):
        return "experiments.baselines"
    if run_id.startswith("ADV"):
        return "experiments.adversarial"
    return "experiments.run"

def stream(i, run_id, dest):
    env = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1", "PYTHONPATH": dest,
           "CUDA_VISIBLE_DEVICES": str(i), "OLLAMA_BASE_URL": f"http://127.0.0.1:{port(i)}"}
    cmd = [sys.executable, "-m", module_for(run_id), f"experiments/configs/paper/{run_id}.yaml"]
    p = subprocess.Popen(cmd, cwd=dest, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env=env, start_new_session=True)
    PROCS[i] = p
    tail = []
    with open(f"{WORK}/logs/{run_id}.log", "a", encoding="utf-8") as log:
        for line in p.stdout:
            log.write(line)
            print(f"[{run_id}] {line}", end="", flush=True)
            tail = (tail + [line])[-40:]
    return p.wait(), "".join(tail)

def note_platform(run_id, commit):
    path = f"{WORK}/platform/{run_id}.json"
    seen = json.load(open(path)) if os.path.exists(path) else []
    seen.append({"platform": "kaggle", "gpu": GPUS[0], "commit": commit,
                 "session_start": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(SESSION_START))})
    json.dump(seen, open(path, "w"), indent=1)

def next_job():
    with LOCK:
        for run_id in KAGGLE_RUNS:
            if run_id in CLAIMED:
                continue
            commit0, have, want, _ = run_status(run_id)
            if have >= want:
                continue
            CLAIMED.add(run_id)
            return run_id, commit0, have, want
    return None

def worker(i):
    say = lambda m: print(f"[gpu {i}] {m}", flush=True)
    while not STOP.is_set():
        job = next_job()
        if job is None:
            say("nothing left for this GPU")
            return
        run_id, commit0, have, want = job
        commit = commit0 or LATEST
        if run_id in NEED_FIXES and not has_fixes(commit):
            PROBLEMS.append(f"{run_id} was started on {commit[:10]}, older than the 11 Nov fixes. Delete "
                            f"finadvisor-results/paper/{run_id}.jsonl from the input and run again.")
            continue
        try:
            dest = code_for(run_id, commit)
            note_platform(run_id, commit)
            say(f"{run_id}: {have}/{want} rows, code {commit[:10]}")
            for attempt in range(1, 4):
                status, tail = stream(i, run_id, dest)
                if status == 0 or STOP.is_set():
                    break
                say(f"*** {run_id} stopped (attempt {attempt} of 3). Ollama log:\n{log_tail(i, 15)}")
                if attempt < 3:
                    restart_server(i)
            else:
                PROBLEMS.append(f"{run_id} failed on GPU {i}. Last output:\n{tail}\nOllama log:\n{log_tail(i)}")
        except (Exception, SystemExit) as e:   # one run's setup failing must not stop this GPU
            PROBLEMS.append(f"{run_id} could not start on GPU {i}: {e}")
        PROCS.pop(i, None)

def watchdog():
    while any(t.is_alive() for t in threads):
        if time.time() > DEADLINE and not STOP.is_set():
            print("\n*** time limit: stopping the runs so this version saves its results", flush=True)
            STOP.set()
            for p in list(PROCS.values()):
                if p.poll() is None:
                    os.killpg(p.pid, signal.SIGTERM)
        time.sleep(20)

threads = [threading.Thread(target=worker, args=(i,), daemon=True) for i in range(len(GPUS))]
for t in threads:
    t.start()
guard = threading.Thread(target=watchdog, daemon=True)
guard.start()
for t in threads:
    t.join()
for path in glob.glob(f"{WORK}/paper/*.jsonl"):
    repair(path)
print("\n" + "\n\n".join(PROBLEMS) if PROBLEMS else "\nno problems")

## 9. Progress

In [ ]:
left = 0
for run_id in KAGGLE_RUNS:
    commit0, have, want, _ = run_status(run_id)
    left += have < want
    print(f"{run_id:22s} {have:4d} / {want}  {'done' if have >= want else ''}")
print("\nALL KAGGLE RUNS FINISHED" if not left else f"\n{left} runs to go: start the next session (see the top)")